In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import gdown
import re
import os
import pandas as pd
import seaborn as sns
import scrublet as scr
import scipy as sp
import matplotlib as mpl
import gc
import scipy

import tables
import scipy.sparse as sp
from scipy.stats import norm
import statsmodels.stats.multitest as multi

import anndata
from typing import Dict, Optional

/nfs/team283/gd11/software/miniconda3/envs/scarches_new/lib/python3.7/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.8) or chardet (5.1.0)/charset_normalizer (2.0.11) doesn't match a supported version!
  RequestsDependencyWarning)


In [ ]:
mpl.rcParams.update(mpl.rcParamsDefault)
%matplotlib inline

In [ ]:
def one_tailed_MAD_test(observed_score, sample_median, sample_mad):
    # calculate z-score
    z_score = (observed_score - sample_median) / sample_mad

    # calculate p-value (one-tailed test)
    p_value = norm.sf(z_score)
    return p_value

In [ ]:
## Additional functions

def dict_from_h5(file: str) -> Dict[str, np.ndarray]:
    """Read in everything from an h5 file and put into a dictionary."""
    d = {}
    with tables.open_file(file) as f:
        # read in everything
        for array in f.walk_nodes("/", "Array"):
            d[array.name] = array.read()
    return d


def _fill_adata_slots_automatically(adata, d):
    """Add other information to the adata object in the appropriate slot."""

    for key, value in d.items():
        try:
            if value is None:
                continue
            value = np.asarray(value)
            if len(value.shape) == 0:
                adata.uns[key] = value
            elif value.shape[0] == adata.shape[0]:
                if (len(value.shape) < 2) or (value.shape[1] < 2):
                    adata.obs[key] = value
                else:
                    adata.obsm[key] = value
            elif value.shape[0] == adata.shape[1]:
                if value.dtype.name.startswith('bytes'):
                    adata.var[key] = value.astype(str)
                else:
                    adata.var[key] = value
            else:
                adata.uns[key] = value
        except Exception:
            print('Unable to load data into AnnData: ', key, value, type(value))

## snRNA reference (raw counts)
# error in original code: case issue with MT counts; added lower call to mt bool loop

## DEPRECATED; PROCESS BY SAMPLE NOW
def load_10x_output(smp_list, metadata=None, type = 'h5', umi_filter=5):

    #Writing output from separate samples, processed using CellRanger, into a dictionary of Scanpy objects:
    ad = {}

    #Generate AnnData for each sample
    for sample_name in smp_list:
            ad[sample_name] = sc.read_10x_h5(sample_name)
            ad[sample_name].var.rename(columns = {'gene_ids':'ENSEMBL'}, inplace = True)
            ad[sample_name].var['SYMBOL'] = ad[sample_name].var.index
            ad[sample_name].var.index = ad[sample_name].var['ENSEMBL']
            ad[sample_name].var.drop(columns=['ENSEMBL'], inplace=True)
            #ad[sample_name].var_names_make_unique()


            sc.pp.calculate_qc_metrics(ad[sample_name], inplace=True)
            ad[sample_name] = ad[sample_name][ad[sample_name].obs['total_counts'] > umi_filter, :]
            ad[sample_name].var['mt'] = [gene.lower().startswith('mt-')
                                         for gene in ad[sample_name].var['SYMBOL']]
            ad[sample_name].obs['mt_frac'] = (ad[sample_name][:,
                   ad[sample_name].var['mt'].tolist()].X.sum(1).A.squeeze()
                                              / ad[sample_name].obs['total_counts'])

            ad[sample_name].obs['sample'] = sample_name
            ad[sample_name].obs['barcode'] = ad[sample_name].obs_names
            ad[sample_name].obs_names = ad[sample_name].obs['sample']+"_"+ad[sample_name].obs['barcode']

    #Merge AnnData objects from all the samples together
    from scipy.sparse import vstack
    stack = vstack([ad[x].X for x in smp_list]) # stack data
    adata = sc.AnnData(stack, var = ad[smp_list[0]].var)
    adata.obs = pd.concat([ad[x].obs for x in smp_list], axis = 0)

    if metadata is not None:
        #Add cleaned metadata to the Anndata.obs table
        obs_merged = pd.merge(left = adata.obs, right = metadata,
                              how = "left", left_on="sample", right_on="SampleID")
        obs_merged.index = obs_merged['sample']+"_"+obs_merged['barcode']
        print(obs_merged.index.equals(adata.obs.index))
        adata.obs = obs_merged
 
    return adata

def process_cellranger(sample, umi_filter=5):
    adata = sc.read_10x_mtx(sample)
    adata.var.rename(columns = {'gene_ids':'ENSEMBL'}, inplace = True)
    adata.var['SYMBOL'] = adata.var.index
    #adata.var.index = adata.var['ENSEMBL']
    #adata.var.drop(columns=['ENSEMBL'], inplace=True)
    adata.var_names_make_unique()


    sc.pp.calculate_qc_metrics(adata, inplace=True)
    adata = adata[adata.obs['total_counts'] > umi_filter, :]
    adata.var['mt'] = [gene.lower().startswith('mt-')
                                         for gene in adata.var['SYMBOL']]
    adata.obs['mt_frac'] = (adata[:,adata.var['mt'].tolist()].X.sum(1).A.squeeze()
                                              / adata.obs['total_counts'])

    adata.obs['sample'] = sample
    adata.obs['barcode'] = adata.obs_names
    adata.obs_names = adata.obs['sample']+"_"+adata.obs['barcode']
    
    return adata

def apply_filters(adata, 
                  threshold_n_genes=500,
                  threshold_min_counts=1000,
                  threshold_mt=0.1, 
                  threshold_total_counts=100000):

    N_genes_filter = np.where(
        adata.obs.n_genes_by_counts > threshold_n_genes, 0, 1)
    min_counts_filter = np.where(
        adata.obs.total_counts > threshold_min_counts, 0, 1)
    pct_counts_mt_filter = np.where(
        adata.obs.mt_frac < threshold_mt, 0, 1)
    total_counts_filter = np.where(
        adata.obs.total_counts < threshold_total_counts, 0, 1)

    total_filtered = np.where(
        (adata.obs.mt_frac < threshold_mt) &
        (adata.obs.total_counts > min_counts_filter) &
        (adata.obs.n_genes_by_counts > threshold_n_genes) &
        (adata.obs.total_counts < threshold_total_counts), 0, 1)
    
    # save filtered information
    filter_summary = pd.DataFrame({'N_genes_filter':N_genes_filter,
                                   'min_total_UMI_filter':min_counts_filter,
                                  'pct_counts_mt_filter':pct_counts_mt_filter,
                                  'total_counts_filter':total_counts_filter,
                                  'total_filtered':total_filtered})\
                        .sum(axis=0)
    
    
    adata = adata[adata.obs.n_genes_by_counts > threshold_n_genes, :]
    adata = adata[adata.obs.total_counts > threshold_min_counts, :]
    adata = adata[adata.obs.mt_frac < threshold_mt, :]
    adata = adata[adata.obs.total_counts < threshold_total_counts, :]
    
    return adata, filter_summary

def run_scrublet(adata):
    if len(adata)>150:
        scrub = scr.Scrublet(adata.X)
        adata.obs['doublet_scores'], adata.obs['predicted_doublets'] = scrub.scrub_doublets()
        # doublet score properly computed; generally around ~0.25
        if hasattr(scrub, 'threshold_'):
            adata.obs['doublet_threshold'] = scrub.threshold_
        else:
            adata.obs['doublet_threshold'] = 0.25
    else:
        # if there is an issue with sample size, just keep all cells
        print('PCA issues likely due to N cells, \
        not detecting doublets')
        adata.obs['doublet_scores'] = 0
        adata.obs['doublet_threshold'] = 1
    
    filtered_doublets = np.where(adata.obs['doublet_scores']<adata.obs['doublet_threshold'],
                                0, 1).sum()
        
    # filter based on detected threshold:
    #adata = adata[adata.obs['doublet_scores']<adata.obs['doublet_threshold'],:]
    
    return adata, filtered_doublets

## Load cell cycle genes
cc_path='../data/metadata/regev_lab_cell_cycle_genes.txt'
def calculate_cycle_score(adata, cc_genes_path = cc_path):
    
    cell_cycle_genes = [x.strip() for x in open(cc_genes_path)]

    # Split into 2 lists
    s_genes = cell_cycle_genes[:43]
    g2m_genes = cell_cycle_genes[43:]
    
    cc_genes = [x for x in cell_cycle_genes if x in adata.var_names]
    print(len(cc_genes))

    adata.layers['counts'] = adata.X.copy()

    # prep for cell cycle ID
    sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
    sc.pp.log1p(adata)
    sc.pp.scale(adata)

    # get cell cycle
    sc.tl.score_genes_cell_cycle(adata, 
                                 s_genes=s_genes, 
                                 g2m_genes=g2m_genes)
    
    return adata

def doublet_analysis(ad, sample, out_dir):
    
    # i.e. scanpy/seurat methods
    sc.pp.normalize_total(ad, target_sum=1e4)
    sc.pp.log1p(ad)
    sc.pp.highly_variable_genes(ad, 
                            min_mean=0.0125, 
                            max_mean=4, 
                            min_disp=0.1)
    selected = ad.var['highly_variable']
    ad = ad[:, selected].copy()
    
    # run scVI on single sample
    scvi.model.SCVI.setup_anndata(
        ad,
        layer="counts",
    )
    vae = scvi.model.SCVI(ad)
    vae.train()

    # run SOLO
    solo = scvi.external.SOLO.from_scvi_model(vae)
    solo.train()
    solo.predict()
    gc.collect()

    # export SOLO results
    scores = solo.predict(soft=True).reset_index(drop=True)
    solo_prediction = solo.predict(soft=False).reset_index(drop=True)
    ad.obs['solo_prediction'] = solo_prediction.tolist()
    ad.obs[['doublet', 'singlet']] = np.array(scores)

    # export posteriors, get latent for UMAP
    latent = vae.get_latent_representation()
    ad.obsm["X_scVI"] = latent

    # use scVI latent space for UMAP generation
    sc.pp.neighbors(ad, use_rep="X_scVI")
    sc.tl.leiden(ad, key_added="leiden_scVI")
    sc.tl.umap(ad, min_dist=0.3)

    # export UMAP for inspection
    sc.pl.umap(
        ad,
        color=["leiden_scVI", "doublet_scores", "doublet", "solo_prediction"],
        ncols=2,
        size=1,
        save = '_{}_doublet_scores.png'.format(sample), vmin=[0, 0, 0, 0],
        frameon=False)

    # export doublet details for each sample
    cell_id = ad.obs['sample'].astype(str)+'#'+ad.obs['barcode'].astype(str)
    ad.obs['cell_id'] = cell_id

    export_cols = ['cell_id', 'doublet_scores', 'doublet_threshold', 'doublet', 'singlet', 
            'solo_prediction', 'leiden_scVI']
    ad.obs[export_cols].to_csv(out_dir+'{}_doublet_calling_results.csv'.format(sample), index=None)

    # export embeddings
    np.save(out_dir+'{}_embeddings.npy'.format(sample), ad.obsm['X_scVI'])

# Load doublet data

In [ ]:
## Load doublets obs and preprocess them
# Read donor metadata
sample_summary = pd.read_csv('../data/samplelist_phase_one.txt', sep='\t', header = None)
sample_summary = sample_summary.rename(columns={0:'sample', 1:'site_id'})

# Load sample directories
doublet_dir = '../data/doublet_analysis/'

file_paths = []
for root, dirs, files in os.walk(doublet_dir, topdown=False):
    for f in files:
        if 'csv' in f:
                file_paths.append(root+f)

In [ ]:
## Testing
cluster_summaries = []
doublet_analysis = []
for f in file_paths:
    doublet_df = pd.read_csv(f)
    sample = re.sub('^.*doublet_analysis/(.*?)_doublet_calling_.*$', '\\1', f)
    doublet_df['cell_id'] = [re.sub('^.*data/(.*?)/cellbender_out/', 
                                    '\\1', i) for i in doublet_df['cell_id']]

    doublet_df['solo_is_doublet'] = np.where(doublet_df['solo_prediction']=='doublet', 1, 0)
    doublet_df['scrublet_default'] = np.where(doublet_df['doublet_scores']>doublet_df['doublet_threshold'], 1, 0)
    doublet_df['scrublet_p90'] = np.where(doublet_df['doublet_scores']>doublet_df['doublet_scores'].quantile(0.9), 1, 0)
    doublet_df['scrublet_p75'] = np.where(doublet_df['doublet_scores']>doublet_df['doublet_scores'].quantile(0.75), 1, 0)
    doublet_df['scrublet_s25'] = np.where(doublet_df['doublet_scores']>0.2, 1, 0)
    
    # get summary info
    cluster_summary = doublet_df.groupby('leiden_scVI', as_index=False)\
        .agg({'solo_is_doublet':'sum', 
              'scrublet_default':'sum',
              'scrublet_p90':'sum',
              'scrublet_p75':'sum',
              'scrublet_s25':'sum',
              'doublet':'median',
              'doublet_scores':'median',
              'cell_id':'count'}).rename(columns={'cell_id':'cell_counts'})
    
    # get method correlation
    corr_cluster = []
    for cluster in sorted(set(doublet_df['leiden_scVI'])):                  
        method_corr, p = scipy.stats.pearsonr(doublet_df['doublet_scores'], doublet_df['doublet'])
        corr_cluster.append(pd.DataFrame({'leiden_scVI':cluster, 'method_corr':method_corr, 'corr_p':p}, index=[0]))
    corr_cluster = pd.concat(corr_cluster, ignore_index=True)
                          
    cluster_summary = cluster_summary.merge(corr_cluster, on = 'leiden_scVI')
    
    # proportions:
    proportions = ['solo_is_doublet', 'scrublet_default', 'scrublet_p90', 'scrublet_p75', 'scrublet_s25']
    for col in proportions:
        cluster_summary[col] = cluster_summary[col].divide(
            doublet_df.groupby('leiden_scVI')['cell_id'].count()
        )

    cluster_summary['sample'] = sample
    doublet_df['sample'] = sample
    cluster_summary = cluster_summary.merge(sample_summary, on = 'sample', how = 'left')
    doublet_df = doublet_df.merge(sample_summary, on = 'sample', how = 'left')
    cluster_summaries.append(cluster_summary)
    doublet_analysis.append(doublet_df)

In [ ]:
# get doublet info
doublet_analysis_df = pd.concat(doublet_analysis, ignore_index=True)
cluster_summaries_df = pd.concat(cluster_summaries, ignore_index=True)

# Load ATAC doublets (optional)

In [ ]:
ATAC_doublets = pd.read_csv(
    "../data/metadata/doublets_ATAC_filtered_only_2023_05.txt",
           index_col = 0, sep = '\t')
ATAC_doublets['cell_id'] = ATAC_doublets.index


# Reload UMAPs (optional)

In [ ]:
# Load sample directories
gbm_directory = '../data/cellbender_input/data/'

file_paths_adata = []
for root, dirs, files in os.walk(gbm_directory, topdown=False):
    for f in files:
        if 'cellbender_out_filtered.h5' in f:
                file_paths_adata.append(root+'/cellbender_out/')

file_paths_scvi = []
for root, dirs, files in os.walk(doublet_dir, topdown=False):
    for f in files:
        if 'npy' in f:
                file_paths_scvi.append(root+f)

In [ ]:
def get_UMAP_from_sample_str(sample_str, 
                             doublet_analysis_df=doublet_analysis_df,
                             size = 1, MAD=False):
    # Load doublet analysis files and adata location
    scvi_file = [i for i in file_paths_scvi if sample_str in i][0]
    sample = [i for i in file_paths_adata if sample_str in i][0]
    sample_id = re.sub('^.*cellranger(.*?)/cellbender_out/$', 
            'cellranger\\1', sample)
    doublet_obs = doublet_analysis_df[doublet_analysis_df['cell_id'].str.contains(sample_str)]
    print(sample_id)

    # Processing raw adata
    adata = process_cellranger(sample)
    adata.obs['sample'] = sample_id
    cell_id = adata.obs['sample'].astype(str)+'#'+adata.obs['barcode'].astype(str)
    adata.obs['cell_id'] = cell_id
    adata, filtered_summary = apply_filters(adata, threshold_total_counts=10000000)

    # Add doublet analysis results to adata
    adata.obs = adata.obs.reset_index().merge(doublet_obs, on = 'cell_id', how = 'left').set_index('index')
    adata.obs = adata.obs.reset_index().merge(ATAC_doublets, on = 'cell_id', how = 'left').set_index('index')
    
    if MAD==True:
        #adata.obs['cluster_id'] = adata.obs['sample']+adata.obs['leiden_scVI'].astype(str)
        clusters = doublet_clusters[doublet_clusters['sample'].str.contains(sample_str)]['leiden_scVI'].tolist()
        adata.obs['MAD_cluster'] = np.where(adata.obs['leiden_scVI'].isin(clusters), 1, 0)
        adata.obs['MAD_individual'] = np.where(adata.obs['cell_id'].isin(
            doublet_analysis_df[(doublet_analysis_df['p_adj']<0.05)]['cell_id']), 1, 0)
    
    adata.obs['leiden_scVI'] = adata.obs['leiden_scVI'].astype(str) 
    adata.obsm['X_scVI'] = np.load(scvi_file)
    sc.pp.neighbors(adata, use_rep="X_scVI")
    sc.tl.umap(adata, min_dist=0.3)

    if MAD==True:
        sc.pl.umap(
            adata,
            color=["leiden_scVI", "doublet_scores", "doublet", "solo_prediction", 
                   "DoubletEnrichment", "DoubletScore", "Filtered",
                   "MAD_cluster", "MAD_individual"],
            ncols=2,
            size=size,
            frameon=False)
    else:
        sc.pl.umap(
                adata,
                color=["leiden_scVI", "doublet_scores", "doublet", "solo_prediction", 
                       "DoubletScore", "DoubletEnrichment", "Filtered"],
                ncols=2,
                size=size, 
                frameon=False)

In [ ]:
# test function
get_UMAP_from_sample_str('arc201_count_2170ae8ab8c2f58272fc8f07c172739d', size = 10, MAD=True)

# Evaluate

- **MAD method (scrublet)** is the actual method, it ends by producing a list of doublet indices (doublet ids) which are used to filter the post-QC anndata

## MAD method (scrublet)

In [ ]:
MAD = doublet_analysis_df.groupby('sample', as_index = False)['doublet_scores'].mad()\
    .rename(columns={'doublet_scores':'doublet_scores_MAD'})
median = doublet_analysis_df.groupby('sample', as_index = False)['doublet_scores'].median()\
    .rename(columns={'doublet_scores':'doublet_scores_median'})
leiden_median = doublet_analysis_df.groupby(['sample', 'leiden_scVI'], 
                                            as_index = False)['doublet_scores'].median()\
    .rename(columns={'doublet_scores':'doublet_scores_leiden_median'})

doublet_analysis_df = doublet_analysis_df.merge(MAD, on = 'sample', how='left')\
                                        .merge(median, on = 'sample', how='left')\
                                        .merge(leiden_median, on = ['sample', 'leiden_scVI'], how='left')

In [ ]:
doublet_analysis_df['p_value'] = one_tailed_MAD_test(doublet_analysis_df['doublet_scores'],
                                                     doublet_analysis_df['doublet_scores_median'], 
                                                     doublet_analysis_df['doublet_scores_MAD'])

In [ ]:
#doublet_analysis_df['p_adj'] = multi.multipletests(doublet_analysis_df['p_value'], method = 'fdr_bh')[1]
# Adjust with BH per sample; only use clusters with observed values > median
p_adj = []
for sample in set(doublet_analysis_df['sample']):
    sample_df = doublet_analysis_df[doublet_analysis_df['sample']==sample]
    sample_df = sample_df[sample_df['doublet_scores']>sample_df['doublet_scores_median']].reset_index(drop=True)
    sample_df['p_adj'] = multi.multipletests(sample_df['p_value'], method = 'fdr_bh')[1]
    p_adj.append(sample_df[['cell_id', 'p_adj']])
p_adj = pd.concat(p_adj)

doublet_analysis_df = doublet_analysis_df.merge(p_adj, on = ['cell_id'], how = 'left').fillna(1)

In [ ]:
cluster_MAD_df = doublet_analysis_df[['sample', 'leiden_scVI', 
                     'doublet_scores_MAD', 'doublet_scores_leiden_median', 
                     'doublet_scores_median']].drop_duplicates()

In [ ]:
cluster_MAD_df['p_value'] = one_tailed_MAD_test(cluster_MAD_df['doublet_scores_leiden_median'],
                                                     cluster_MAD_df['doublet_scores_median'], 
                                                     cluster_MAD_df['doublet_scores_MAD'])

In [ ]:
#cluster_MAD_df['p_adj'] = multi.multipletests(cluster_MAD_df['p_value'], method = 'fdr_bh')[1]

In [ ]:
# Adjust with BH per sample; only use clusters with observed values > median
p_adj = []
for sample in set(cluster_MAD_df['sample']):
    sample_df = cluster_MAD_df[cluster_MAD_df['sample']==sample]
    sample_df = sample_df[sample_df['doublet_scores_leiden_median']>sample_df['doublet_scores_median']]\
    .reset_index(drop=True)
    sample_df['p_adj'] = multi.multipletests(sample_df['p_value'], method = 'fdr_bh')[1]
    p_adj.append(sample_df[['sample', 'leiden_scVI', 'p_adj']])
p_adj = pd.concat(p_adj)

cluster_MAD_df = cluster_MAD_df.merge(p_adj, on = ['sample', 'leiden_scVI'], how = 'left').fillna(1)


### Define thresholds

#### FDR (BH)

In [ ]:
doublet_clusters = cluster_MAD_df[cluster_MAD_df['p_adj']<0.1].copy()
doublet_clusters['cluster_id'] = doublet_clusters['sample']+'|'+doublet_clusters['leiden_scVI'].astype(str)
doublet_analysis_df['cluster_id'] = doublet_analysis_df['sample']+'|'+doublet_analysis_df['leiden_scVI'].astype(str)


In [ ]:
doublet_ids = doublet_analysis_df[(doublet_analysis_df['p_adj']<0.01) |
                                  (doublet_analysis_df['cluster_id'].isin(doublet_clusters['cluster_id']))]

In [ ]:
cluster_summaries_df['MAD_adj'] = np.where(cluster_summaries_df['cluster_id'].isin(doublet_clusters['cluster_id']),
                                       1, 0)

# Finalise adata

In [ ]:
adata = sc.read_h5ad('../data/\
adata/GBM_LEAP_23_05_04_QC_no_doublet_removal.h5ad')

In [ ]:
# add patient metadata
patient_metadata = pd.read_csv('../data/metadata/phase_one_metadata.txt',
                              sep = '\t')
patient_cols = ['donor_id', '# of blocks', 'Sex', 'Age', 'Ethnicity ', 'Weight_kg', 
                 'BMI_kg_m2', 'Tumour_spatial_information']

experiment_dates = pd.read_csv('../data/metadata/phase_one_experiment_dates.txt',
                              sep = '\t')
experiment_dates = experiment_dates.drop_duplicates().reset_index(drop=True)

adata.obs = adata.obs.reset_index().merge(patient_metadata[patient_cols], 
                                          on = 'donor_id', how = 'left').set_index('index')

adata.obs = adata.obs.reset_index().merge(experiment_dates, 
                              on = 'site_id', how = 'left').set_index('index')

# apply final filters:
adata = adata[adata.obs['total_counts']<75000]

# filter bad ATAC barcodes
good_ATAC_barcodes = pd.read_csv('../data/metadata/\
barcodes_ATAC_filtered_only_2023_05.txt', sep='/t')
adata = adata[adata.obs['cell_id'].isin(good_ATAC_barcodes.index)]


/nfs/team283/gd11/software/miniconda3/envs/scarches_new/lib/python3.7/site-packages/pandas/util/_decorators.py:311: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  return func(*args, **kwargs)


In [ ]:
adata[~adata.obs['cell_id'].isin(doublet_ids['cell_id'])]

View of AnnData object with n_obs × n_vars = 1041869 × 36601
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mt_frac', 'sample', 'barcode', 'doublet_scores', 'doublet_threshold', 'n_counts', 'S_score', 'G2M_score', 'phase', 'site_id', 'donor_id', 'cell_id', '# of blocks', 'Sex', 'Age', 'Ethnicity ', 'Weight_kg', 'BMI_kg_m2', 'Tumour_spatial_information', 'date'
    var: 'ENSEMBL', 'feature_types', 'SYMBOL', 'mt'
    layers: 'counts'

In [ ]:
add_doublet_cols = ['cell_id', 'doublet', 'singlet', 'solo_prediction', 'leiden_scVI']

adata = adata[~adata.obs['cell_id'].isin(doublet_ids['cell_id'])]

In [ ]:
adata.obs = adata.obs.reset_index().merge(doublet_analysis_df[add_doublet_cols], 
           on = 'cell_id', how = 'left').set_index('index')\
            .rename(columns={'leiden_scVI':'leiden_scVI_per_reaction'})

In [ ]:
adata.obs

,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,mt_frac,sample,...,Age,Ethnicity,Weight_kg,BMI_kg_m2,Tumour_spatial_information,date,doublet,singlet,solo_prediction,leiden_scVI_per_reaction
index,,,,,,,,,,,,,,,,,,,,,
/lustre/scratch126/cellgen/team283/gd11/gd11/data_GBM/cellbender_input/data/cellranger-arc201_count_adf395d2259f5577779c070f7ebe5b76/cellbender_out/_CATTATCTCATTCATC-1,9592,9.168789,74322.0,11.216176,21.008584,26.922042,33.950916,46.342940,0.058435,cellranger-arc201_count_adf395d2259f5577779c07...,...,62,NaN,80.0,25.2,Right Fronto-Temporal Lobe,27/05/2022,1.536253,-1.218887,doublet,10
/lustre/scratch126/cellgen/team283/gd11/gd11/data_GBM/cellbender_input/data/cellranger-arc201_count_adf395d2259f5577779c070f7ebe5b76/cellbender_out/_GGTTATATCCTAACGG-1,11215,9.325097,73345.0,11.202943,15.388915,21.472493,28.544550,40.541278,0.012721,cellranger-arc201_count_adf395d2259f5577779c07...,...,62,NaN,80.0,25.2,Right Fronto-Temporal Lobe,27/05/2022,1.202547,-0.405429,doublet,13
/lustre/scratch126/cellgen/team283/gd11/gd11/data_GBM/cellbender_input/data/cellranger-arc201_count_adf395d2259f5577779c070f7ebe5b76/cellbender_out/_GGTTATATCTGTAATG-1,10725,9.280426,70302.0,11.160570,15.838810,20.936815,27.555404,39.481096,0.062004,cellranger-arc201_count_adf395d2259f5577779c07...,...,62,NaN,80.0,25.2,Right Fronto-Temporal Lobe,27/05/2022,0.864324,-1.237087,doublet,17
/lustre/scratch126/cellgen/team283/gd11/gd11/data_GBM/cellbender_input/data/cellranger-arc201_count_adf395d2259f5577779c070f7ebe5b76/cellbender_out/_TCGCGAGGTCAAAGGG-1,10531,9.262174,68521.0,11.134911,21.581705,26.785949,32.988427,43.888735,0.093081,cellranger-arc201_count_adf395d2259f5577779c07...,...,62,NaN,80.0,25.2,Right Fronto-Temporal Lobe,27/05/2022,0.720737,-1.030397,doublet,7
/lustre/scratch126/cellgen/team283/gd11/gd11/data_GBM/cellbender_input/data/cellranger-arc201_count_adf395d2259f5577779c070f7ebe5b76/cellbender_out/_GCCTACTTCTATGACA-1,9967,9.207135,66839.0,11.110057,19.991322,25.194871,31.460674,42.521582,0.076572,cellranger-arc201_count_adf395d2259f5577779c07...,...,62,NaN,80.0,25.2,Right Fronto-Temporal Lobe,27/05/2022,1.486328,-0.526009,doublet,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
/lustre/scratch126/cellgen/team283/gd11/gd11/data_GBM/cellbender_input/data/cellranger-arc201_count_3c475e5faf73bf8c98e14aeb2ca0ed42/cellbender_out/_CAATCCCTCCTTCTAG-1,769,6.646391,1003.0,6.911747,19.641077,29.611167,43.270189,73.180459,0.016949,cellranger-arc201_count_3c475e5faf73bf8c98e14a...,...,70,White-British,85.0,24.0,Right Temporal Lobe with Right Frontal Lobe Ex...,11/08/2022,-5.970526,4.595556,singlet,1
/lustre/scratch126/cellgen/team283/gd11/gd11/data_GBM/cellbender_input/data/cellranger-arc201_count_3c475e5faf73bf8c98e14aeb2ca0ed42/cellbender_out/_CCTGACTTCAATGAGG-1,631,6.448889,1005.0,6.913737,29.950249,41.592040,57.114428,86.965174,0.003980,cellranger-arc201_count_3c475e5faf73bf8c98e14a...,...,70,White-British,85.0,24.0,Right Temporal Lobe with Right Frontal Lobe Ex...,11/08/2022,-7.729255,6.461615,singlet,1
/lustre/scratch126/cellgen/team283/gd11/gd11/data_GBM/cellbender_input/data/cellranger-arc201_count_3c475e5faf73bf8c98e14aeb2ca0ed42/cellbender_out/_CCGTTGCGTCCTAATC-1,693,6.542472,1003.0,6.911747,23.828514,34.995015,50.847458,80.757727,0.001994,cellranger-arc201_count_3c475e5faf73bf8c98e14a...,...,70,White-British,85.0,24.0,Right Temporal Lobe with Right Frontal Lobe Ex...,11/08/2022,-6.325414,4.996946,singlet,1


In [ ]:
adata.write_h5ad('../data/adata/GBM_LEAP_23_05_15_MAD.h5ad')